# Fixed Search-Query Fanout Scaling Analysis ($k \in \{1, 2, 4, 8\}$)

This notebook provides visual analysis of empirical results from the **Fixed Fanout Scaling Experiment** (`fixed_fanout_scaling_v1`).

### Overview & Hypotheses
1. **Quality Gain**: Increasing candidate search query fanout depth ($k=1 \to 2 \to 4 \to 8$) systematically improves response intent satisfaction, specificity, and evidence groundedness.
2. **Hallucination Reduction**: Broader domain coverage significantly decreases unsupported claim risk.
3. **Quality-Cost Frontier**: Identifies the Pareto optimal fanout depth balancing API costs, context length, latency, and quality.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set figure aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.labelweight": "bold",
    "figure.titlesize": 15,
    "figure.titleweight": "bold",
})

# Robust path resolution to locate repository root
cwd = os.getcwd()
if os.path.basename(cwd) == "notebooks":
    PROJECT_ROOT = os.path.dirname(cwd)
elif os.path.exists(os.path.join(cwd, "outputs/fixed_fanout_scaling_v1")):
    PROJECT_ROOT = cwd
else:
    # Fallback search upwards for outputs/ directory
    curr = cwd
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, "outputs/fixed_fanout_scaling_v1")):
            PROJECT_ROOT = curr
            break
        curr = os.path.dirname(curr)

FRONTIER_CSV = os.path.join(PROJECT_ROOT, "outputs/fixed_fanout_scaling_v1/quality_cost_frontier.csv")
MARGINAL_CSV = os.path.join(PROJECT_ROOT, "outputs/fixed_fanout_scaling_v1/marginal_gains.csv")

df_frontier = pd.read_csv(FRONTIER_CSV)
df_marginal = pd.read_csv(MARGINAL_CSV)

print(f"Loaded Frontier CSV ({len(df_frontier)} rows) & Marginal CSV ({len(df_marginal)} rows) from:\n  - {FRONTIER_CSV}")

In [ ]:
# Filter overall method results
df_overall = df_frontier[df_frontier["analysis_group"] == "overall"].sort_values("requested_k")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Final Quality Scores vs k
k_vals = df_overall["requested_k"]
axes[0].plot(k_vals, df_overall["final_intent_satisfaction_mean"], marker="o", linewidth=2.5, markersize=8, label="Intent Satisfaction", color="#2b5c8f")
axes[0].plot(k_vals, df_overall["final_specificity_mean"], marker="s", linewidth=2.5, markersize=8, label="Specificity", color="#2a9d8f")
axes[0].plot(k_vals, df_overall["final_non_genericness_mean"], marker="^", linewidth=2.5, markersize=8, label="Non-Genericness", color="#e76f51")
axes[0].plot(k_vals, df_overall["final_groundedness_mean"], marker="D", linewidth=2.5, markersize=8, label="Evidence Groundedness", color="#9c89b8")

axes[0].set_xticks([1, 2, 4, 8])
axes[0].set_xlabel("Fanout Depth (k)")
axes[0].set_ylabel("Mean Humanoid Quality Score (1-5)")
axes[0].set_title("Response Quality Scaling Across Fanout Depth")
axes[0].set_ylim(2.5, 5.1)
axes[0].legend(frameon=True, facecolor="white", edgecolor="none")
axes[0].grid(True, linestyle="--", alpha=0.6)

# Annotate Intent Satisfaction values
for k, val in zip(k_vals, df_overall["final_intent_satisfaction_mean"]):
    axes[0].annotate(f"{val:.2f}", (k, val), textcoords="offset points", xytext=(0, 10), ha="center", weight="bold")

# Panel B: Unsupported Claim Risk (Hallucination Risk)
axes[1].plot(k_vals, df_overall["final_unsupported_claim_risk_mean"], marker="v", linewidth=3, markersize=9, color="#e63946", label="Unsupported Claim Risk")
axes[1].set_xticks([1, 2, 4, 8])
axes[1].set_xlabel("Fanout Depth (k)")
axes[1].set_ylabel("Risk Score (Lower is Better)")
axes[1].set_title("Hallucination Risk Reduction vs. Fanout Depth")
axes[1].set_ylim(0.8, 2.1)
axes[1].legend(frameon=True, facecolor="white", edgecolor="none")
axes[1].grid(True, linestyle="--", alpha=0.6)

for k, val in zip(k_vals, df_overall["final_unsupported_claim_risk_mean"]):
    axes[1].annotate(f"{val:.2f}", (k, val), textcoords="offset points", xytext=(0, 10), ha="center", weight="bold", color="#e63946")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Unique URLs & Unique Domains
axes[0].bar(k_vals - 0.2, df_overall["mean_unique_urls"], width=0.4, label="Unique URLs Retrieved", color="#457b9d")
axes[0].bar(k_vals + 0.2, df_overall["mean_unique_domains"], width=0.4, label="Unique Domains Reached", color="#a8dadc")
axes[0].set_xticks([1, 2, 4, 8])
axes[0].set_xlabel("Fanout Depth (k)")
axes[0].set_ylabel("Count")
axes[0].set_title("Information Retrieval Breadth Across k")
axes[0].legend(frameon=True)
axes[0].grid(True, linestyle="--", alpha=0.6)

for k, val in zip(k_vals, df_overall["mean_unique_urls"]):
    axes[0].annotate(f"{val:.1f}", (k - 0.2, val + 0.5), ha="center", size=9, weight="bold")
for k, val in zip(k_vals, df_overall["mean_unique_domains"]):
    axes[0].annotate(f"{val:.1f}", (k + 0.2, val + 0.5), ha="center", size=9, weight="bold")

# Panel B: Synthesis Context Length vs Total Latency
ax2 = axes[1].twinx()
p1 = axes[1].plot(k_vals, df_overall["mean_synthesis_context_size"] / 1000, marker="o", color="#6a0572", linewidth=2.5, label="Context Size (kChars)")
p2 = ax2.plot(k_vals, df_overall["mean_total_latency"], marker="s", color="#f72585", linewidth=2.5, linestyle="--", label="Total Latency (s)")

axes[1].set_xticks([1, 2, 4, 8])
axes[1].set_xlabel("Fanout Depth (k)")
axes[1].set_ylabel("Synthesis Context Size (Thousand Chars)", color="#6a0572")
ax2.set_ylabel("Total Latency (Seconds)", color="#f72585")
axes[1].set_title("Resource Consumption & Latency Trade-Off")

# Merge legends
lines = p1 + p2
labels = [l.get_label() for l in lines]
axes[1].legend(lines, labels, loc="upper left", frameon=True)
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Filter key marginal gain comparisons
key_comp = ["k1_to_k2", "k2_to_k4", "k4_to_k8", "k1_to_k8"]
metrics_to_plot = ["final_intent_satisfaction", "final_non_genericness", "retrieval_constraint_coverage", "final_groundedness"]

df_m_sub = df_marginal[(df_marginal["comparison"].isin(key_comp)) & (df_marginal["metric"].isin(metrics_to_plot))]

plt.figure(figsize=(10, 5))
chart = sns.barplot(
    data=df_m_sub,
    x="metric",
    y="mean_paired_diff",
    hue="comparison",
    palette="Blues_d",
    edgecolor="black"
)

plt.title("Paired Marginal Quality Gains Across Fanout Depth Steps")
plt.xlabel("Quality Metric")
plt.ylabel("Mean Paired Difference (Delta)")
plt.xticks(rotation=15, ha="right")
plt.axhline(0, color="gray", linewidth=1, linestyle="--")
plt.legend(title="Step Comparison", frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df_overall,
    x="mean_total_latency",
    y="final_intent_satisfaction_mean",
    size="requested_k",
    sizes=(100, 400),
    hue="requested_k",
    palette="viridis",
    legend="brief"
)

# Connect Pareto points
plt.plot(df_overall["mean_total_latency"], df_overall["final_intent_satisfaction_mean"], linestyle="--", color="gray", alpha=0.7)

for _, row in df_overall.iterrows():
    plt.annotate(
        f"k={int(row['requested_k'])}\n({row['final_intent_satisfaction_mean']:.2f})",
        (row["mean_total_latency"], row["final_intent_satisfaction_mean"]),
        textcoords="offset points",
        xytext=(15, -5),
        weight="bold"
    )

plt.title("Quality-Cost Pareto Frontier (Intent Satisfaction vs. Latency)")
plt.xlabel("Total Latency (Seconds)")
plt.ylabel("Intent Satisfaction (1-5 Scale)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Summary of Key Findings

1. **Monotonic Quality Gains**: Increasing candidate fanout from $k=1 \to k=8$ drives significant improvements in **Intent Satisfaction (+1.05)** and **Response Specificity (+1.19)**.
2. **Hallucination Mitigation**: Broader domain retrieval cuts **Unsupported Claim Risk by 40%** (1.79 $\to$ 1.08).
3. **Pareto Sweet Spot ($k=4$)**:
   - $k=4$ captures **~72% of full quality gains** while avoiding extreme context growth (16.8k chars vs. 32.0k chars at $k=8$) and keeping overall latency low (36.2s vs. 40.6s).